# 02 — Time Travel & Schema Evolution: Wenn der Anbieter sein Format ändert

Szenario: Der ESG-Datenanbieter NZDPU liefert ein API-Update. Felder wurden umbenannt, neue Felder kamen dazu, die Struktur hat sich leicht verändert. Wir schauen uns an was Iceberg damit macht — und warum bestehende Reports trotzdem weiterlaufen.

In [9]:
import sys
sys.path.insert(0, "/home/jovyan/notebooks")
from spark_init import get_spark_session, trino_query, show

spark = get_spark_session("02-time-travel")

---
## Ausgangslage: Aktueller Zustand der NZDPU-Tabelle

Die Tabelle wurde per `make seed` aus dem NZDPU-JSON befüllt. Schauen wir uns das Schema und einige Beispielzeilen an.

In [2]:
df_current = spark.sql("SELECT * FROM nessie.raw.nzdpu_emissions")
print(f"Zeilen: {df_current.count()}")
print("Schema:")
df_current.printSchema()
df_current.show(5, truncate=False)

Zeilen: 90
Schema:
root
 |-- company_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- isin: string (nullable = true)
 |-- lei: string (nullable = true)
 |-- country: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- reporting_year: long (nullable = true)
 |-- scope_1_tco2e: long (nullable = true)
 |-- scope_2_location_tco2e: long (nullable = true)
 |-- scope_2_market_tco2e: long (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- reporting_framework: string (nullable = true)

+----------+------------+------------+----------------+-------+-----------+--------------+-------------+----------------------+--------------------+--------------------+-------------------+
|company_id|company_name|isin        |lei             |country|sector     |reporting_year|scope_1_tco2e|scope_2_location_tco2e|scope_2_market_tco2e|verification_status |reporting_framework|
+----------+------------+------------+----------------+-------+-------

In [3]:
spark.sql("""
    SELECT snapshot_id,
           date_format(committed_at, 'yyyy-MM-dd HH:mm:ss') AS committed_at,
           operation
    FROM nessie.raw.nzdpu_emissions.snapshots
    ORDER BY committed_at
""").show(truncate=False)

baseline_snapshot = spark.sql("""
    SELECT snapshot_id FROM nessie.raw.nzdpu_emissions.snapshots
    ORDER BY committed_at DESC LIMIT 1
""").collect()[0][0]

print(f"\n\U0001f4cc Baseline Snapshot: {baseline_snapshot}")

+-------------------+-------------------+---------+
|snapshot_id        |committed_at       |operation|
+-------------------+-------------------+---------+
|7242930962477831529|2026-04-12 20:18:12|append   |
+-------------------+-------------------+---------+


📌 Baseline Snapshot: 7242930962477831529


---
## Bestehender Trino-Report

Ein einfacher Report der auf der NZDPU-Tabelle basiert. Der läuft produktiv und darf nicht brechen.

In [4]:
report_sql = """
    SELECT sector,
           count(*) AS unternehmen,
           round(avg(scope_1_tco2e), 0) AS avg_scope1_tco2e,
           round(avg(scope_2_location_tco2e), 0) AS avg_scope2_tco2e
    FROM raw.nzdpu_emissions
    GROUP BY sector
    ORDER BY avg_scope1_tco2e DESC
"""

print("\U0001f4ca Report: Durchschnittliche Emissionen nach Sektor")
report_baseline = trino_query(report_sql)
print(report_baseline)

📊 Report: Durchschnittliche Emissionen nach Sektor
                   sector  unternehmen  avg_scope1_tco2e  avg_scope2_tco2e
0                  Energy            9        42608363.0          804036.0
1               Materials            6        12189662.0         1116520.0
2  Consumer Discretionary           15         3977111.0          384249.0
3             Industrials            9         3382378.0          459361.0
4        Consumer Staples            6         1412384.0          272030.0
5             Health Care           15          774020.0          186913.0
6  Communication Services            3          588197.0          842796.0
7  Information Technology            9          115811.0          390581.0
8              Financials           18           75769.0           59201.0


---
## Die neue Lieferung: API-Format hat sich umgebaut

Der Anbieter hat seine API von Grund auf neu strukturiert. Schauen wir uns erst an wie sich das Format konkret verändert hat.

In [5]:
import json

with open("/home/jovyan/data/sample/nzdpu_emissions.json") as f:
    v1 = json.load(f)
with open("/home/jovyan/data/sample/nzdpu_emissions_v2.json") as f:
    v2 = json.load(f)

# V1: {data: [{company, reporting_periods: [...]}]} — eine Ebene pro Unternehmen
v1_sample = v1["data"][0].copy()
v1_sample["reporting_periods"] = [v1_sample["reporting_periods"][0]]

print("=== V1 Format (bisherig): verschachtelt pro Unternehmen ===")
print(json.dumps(v1_sample, indent=2)[:700])

print()
print("=== V2 Format (neu): flaches Array, eine Zeile pro Unternehmen x Jahr ===")
print(json.dumps(v2[0], indent=2))


=== V1 Format (bisherig): verschachtelt pro Unternehmen ===
{
  "company_id": "NZDPU-0001",
  "company_name": "Siemens AG",
  "isin": "DE0007236101",
  "lei": "549300DE00072361",
  "country_of_incorporation": "Germany",
  "primary_sector": "Industrials",
  "reporting_periods": [
    {
      "reporting_year": 2021,
      "reporting_framework": null,
      "verification_status": "third_party_verified",
      "scope_1": {
        "value": 3430896,
        "unit": "tCO2e"
      },
      "scope_2_location_based": {
        "value": 381067,
        "unit": "tCO2e"
      },
      "scope_2_market_based": {
        "value": 270562,
        "unit": "tCO2e"
      },
      "scope_3": null,
      "net_zero_target": null
    }
  ],
  "_meta": {
    "source": "NZDP

=== V2 Format (neu): flaches Array, eine Zeile pro Unternehmen x Jahr ===
{
  "entity": {
    "id": "NZDPU-0001",
    "name": "Siemens AG",
    "isin": "DE0007236101",
    "lei": "549300DE00072361",
    "country": "Germany"
  },
  "indust

Was sich geändert hat:

| Feld (V1) | Feld (V2) | Änderung |
|---|---|---|
| `company_name`, `country_of_incorporation` | `entity.name`, `entity.country` | In `entity`-Objekt verschachtelt |
| `primary_sector` | `industry_classification` | Umbenannt |
| `scope_1.value`, `scope_2_location_based.value` | `emissions.scope_1_tco2e`, `emissions.scope_2_location_tco2e` | In `emissions`-Objekt, flache Zahlen |
| *(nicht vorhanden)* | `emissions.scope_3_total_tco2e` | **Neu**: Scope-3-Gesamtemissionen |
| `net_zero_target.target_year` | `climate_target.net_zero_year` | Vereinfacht, umbenannt |
| `verification_status` (in Period) | `metadata.verification` | In `metadata`-Objekt |

Das ist die Realität bei API-basierten Datenquellen. Wir müssen das flattened in unsere bestehende Tabelle bekommen — ohne die bestehenden Daten anzutasten.

In [6]:
# V2 JSON mit Spark laden — verschachtelte Struktur direkt sichtbar
df_v2_raw = spark.read.option("multiLine", "true").json("/home/jovyan/data/sample/nzdpu_emissions_v2.json")

print(f"V2 Eintraege: {df_v2_raw.count()} (ein Record pro Unternehmen x Berichtsjahr)")
print()
print("Schema (verschachtelt — wie die neue API es liefert):")
df_v2_raw.printSchema()


V2 Eintraege: 90 (ein Record pro Unternehmen x Berichtsjahr)

Schema (verschachtelt — wie die neue API es liefert):
root
 |-- climate_target: struct (nullable = true)
 |    |-- net_zero_year: long (nullable = true)
 |-- emissions: struct (nullable = true)
 |    |-- scope_1_tco2e: long (nullable = true)
 |    |-- scope_2_location_tco2e: long (nullable = true)
 |    |-- scope_2_market_tco2e: long (nullable = true)
 |    |-- scope_3_total_tco2e: long (nullable = true)
 |    |-- unit: string (nullable = true)
 |-- entity: struct (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- isin: string (nullable = true)
 |    |-- lei: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- industry_classification: string (nullable = true)
 |-- metadata: struct (nullable = true)
 |    |-- api_version: string (nullable = true)
 |    |-- reporting_framework: string (nullable = true)
 |    |-- verification: string (nullable = true

In [7]:
from pyspark.sql.functions import col

# Flattening: nested -> tabular, Feldnamen auf internes Schema mappen
# "industry_classification" -> "sector" (interne Bezeichnung)
# "metadata.verification"   -> "verification_status"
df_to_append = df_v2_raw.select(
    col("entity.id").alias("company_id"),
    col("entity.name").alias("company_name"),
    col("entity.isin").alias("isin"),
    col("entity.lei").alias("lei"),
    col("entity.country").alias("country"),
    col("industry_classification").alias("sector"),
    col("reporting_year"),
    col("emissions.scope_1_tco2e").alias("scope_1_tco2e"),
    col("emissions.scope_2_location_tco2e").alias("scope_2_location_tco2e"),
    col("emissions.scope_2_market_tco2e").alias("scope_2_market_tco2e"),
    col("metadata.verification").alias("verification_status"),
    col("metadata.reporting_framework").alias("reporting_framework"),
    col("climate_target.net_zero_year").cast("int").alias("net_zero_target_year"),
    col("emissions.scope_3_total_tco2e").alias("scope_3_total_tco2e"),
)

print("Nach Flattening:")
df_to_append.show(5, truncate=False)
cnt = df_to_append.count()
print(f"{cnt} Eintraege bereit — davon einige mit scope_3 und net_zero_year gefuellt.")


Nach Flattening:
+----------+------------+------------+----------------+-------+-----------+--------------+-------------+----------------------+--------------------+--------------------+-------------------+--------------------+-------------------+
|company_id|company_name|isin        |lei             |country|sector     |reporting_year|scope_1_tco2e|scope_2_location_tco2e|scope_2_market_tco2e|verification_status |reporting_framework|net_zero_target_year|scope_3_total_tco2e|
+----------+------------+------------+----------------+-------+-----------+--------------+-------------+----------------------+--------------------+--------------------+-------------------+--------------------+-------------------+
|NZDPU-0001|Siemens AG  |DE0007236101|549300DE00072361|Germany|Industrials|2021          |3430896      |381067                |270562              |third_party_verified|NULL               |NULL                |NULL               |
|NZDPU-0001|Siemens AG  |DE0007236101|549300DE00072361|Germ

---
## Schema Evolution — neue Felder aufnehmen

Iceberg kann das Schema erweitern ohne bestehende Daten zu verändern. Neue Felder werden hinzugefügt, alte Daten haben dort `NULL`.

Das ist eine **reine Metadaten-Operation** — kein einziger Parquet-Block wird umgeschrieben.

In [ ]:
spark.sql("ALTER TABLE nessie.raw.nzdpu_emissions ADD COLUMNS (net_zero_target_year INT)")
print("\u2705 net_zero_target_year hinzugefügt")

spark.sql("ALTER TABLE nessie.raw.nzdpu_emissions ADD COLUMNS (scope_3_total_tco2e BIGINT)")
print("\u2705 scope_3_total_tco2e hinzugefügt")

print("\nNeues Schema:")
spark.sql("DESCRIBE nessie.raw.nzdpu_emissions").show()

In [ ]:
print("Bestehende Daten: neue Felder sind NULL (korrekt — Felder existierten bei der Erfassung nicht):")
spark.sql("""
    SELECT company_name, sector, reporting_year,
           net_zero_target_year, scope_3_total_tco2e
    FROM nessie.raw.nzdpu_emissions
    LIMIT 5
""").show(truncate=False)

### Feld-Mapping beim Schreiben

Der Anbieter verwendet `industry_classification`, wir intern `sector`.
Das Mapping haben wir bereits beim Flattening gemacht (`.alias("sector")`).

Ebenso wurde `metadata.verification` → `verification_status` gemappt.
Die bestehende Spalte `sector` bleibt erhalten — der Report ist nicht betroffen.

In [ ]:
# df_to_append hat bereits das korrekte Schema (14 Spalten).
# Spaltenreihenfolge exakt an die Tabelle anpassen und appenden.
table_columns = spark.sql("SELECT * FROM nessie.raw.nzdpu_emissions LIMIT 0").columns
df_to_append.select(table_columns).writeTo("nessie.raw.nzdpu_emissions").append()

new_count = spark.sql("SELECT count(*) FROM nessie.raw.nzdpu_emissions").collect()[0][0]
print(f" Neue Lieferung geschrieben. Zeilen jetzt: {new_count}")

---
## Snapshot-Historie: Was ist passiert?

Jede Schreiboperation hat einen neuen Snapshot erzeugt. Schauen wir uns die Liste an.

In [ ]:
spark.sql("""
    SELECT snapshot_id,
           date_format(committed_at, 'yyyy-MM-dd HH:mm:ss') AS committed_at,
           operation,
           summary['added-records']    AS added_records,
           summary['total-records']    AS total_records,
           summary['added-data-files'] AS added_files
    FROM nessie.raw.nzdpu_emissions.snapshots
    ORDER BY committed_at
""").show(truncate=False)

print("Snapshot 1: Ursprünglicher Ingest (make seed)")
print("Snapshot 2: Append der neuen Lieferung")
print("(Schema-Änderungen via ALTER TABLE erzeugen keinen eigenen Daten-Snapshot)")

---
## Time Travel: Alte und neue Daten vergleichen

Mit dem Baseline-Snapshot können wir jederzeit den Zustand **vor** der Schema-Änderung und dem Append lesen — ohne Backup, ohne Restore.

In [ ]:
print(f"\U0001f4cc Daten zum Zeitpunkt des Baseline Snapshots ({baseline_snapshot}):")
df_old = spark.sql(f"SELECT * FROM nessie.raw.nzdpu_emissions VERSION AS OF {baseline_snapshot}")
print(f"Zeilen: {df_old.count()}")
print("Schema (ohne net_zero_target_year und scope_3_total_tco2e):")
df_old.printSchema()
df_old.show(3, truncate=False)

In [ ]:
print("Aktuelle Daten (nach Schema Evolution + Append):")
df_now = spark.sql("SELECT * FROM nessie.raw.nzdpu_emissions")
print(f"Zeilen: {df_now.count()}")

print("\nAlte Zeilen (net_zero_target_year = NULL, scope_3_total_tco2e = NULL):")
spark.sql("""
    SELECT company_name, sector, reporting_year, net_zero_target_year, scope_3_total_tco2e
    FROM nessie.raw.nzdpu_emissions
    WHERE net_zero_target_year IS NULL
    LIMIT 3
""").show(truncate=False)

print("Neue Zeilen (net_zero_target_year gefüllt):")
spark.sql("""
    SELECT company_name, sector, reporting_year, net_zero_target_year, scope_3_total_tco2e
    FROM nessie.raw.nzdpu_emissions
    WHERE net_zero_target_year IS NOT NULL
    LIMIT 3
""").show(truncate=False)

---
## Der Trino-Report läuft weiter — unverändert

Der Report von vorhin nutzt `sector`, `scope_1_tco2e` und `scope_2_location_tco2e` — diese Felder existieren weiterhin. Neue Felder stören ihn nicht.

In [ ]:
print("\U0001f4ca Report NACH Schema Evolution + neuem Append:")
report_after = trino_query(report_sql)
print(report_after)

print("\n\U0001f4ca Report VORHER (Baseline):")
print(report_baseline)

print("\n\u2705 Der Report nutzt 'sector' — das existiert weiterhin.")
print("Die neuen Felder (net_zero_target_year, scope_3_total_tco2e) stören nicht.")
print("Bestehende Queries brechen nicht durch Schema Evolution.")

In [ ]:
report_old_sql = f"""
    SELECT sector,
           count(*) AS unternehmen,
           round(avg(scope_1_tco2e), 0) AS avg_scope1_tco2e
    FROM raw.nzdpu_emissions FOR VERSION AS OF {baseline_snapshot}
    GROUP BY sector
    ORDER BY avg_scope1_tco2e DESC
"""

print("\U0001f4ca Report explizit auf Baseline Snapshot:")
print(trino_query(report_old_sql))
print("\nFalls nötig: jederzeit den exakten alten Zustand reproduzieren.")

---
## Iceberg unter der Haube: Was hat sich auf dem Storage geändert?

Schauen wir uns an was auf MinIO passiert ist — welche Parquet-Dateien jetzt existieren.

In [ ]:
spark.sql("""
    SELECT file_path,
           record_count,
           round(file_size_in_bytes / 1024.0, 1) AS size_kb
    FROM nessie.raw.nzdpu_emissions.files
""").show(truncate=False)

print("Iceberg hat KEINE bestehenden Dateien verändert.")
print("Die neuen Daten liegen in einer neuen Parquet-Datei.")
print("Alter Snapshot zeigt auf alte Datei \u2192 deshalb funktioniert Time Travel.")

---
## Schema-Operationen in Iceberg: Alles Metadaten

Bonus: Iceberg kann Spalten umbenennen — ebenfalls ohne Daten-Rewrite.

Parquet-Dateien referenzieren Spalten intern über **Feld-IDs**, nicht über Namen. Deshalb kann Iceberg einen Spaltennamen in den Metadaten ändern, ohne die Datendateien anfassen zu müssen.

In [ ]:
print("Bonus: Spalte umbenennen — reine Metadaten-Operation:")
spark.sql("ALTER TABLE nessie.raw.nzdpu_emissions RENAME COLUMN scope_3_total_tco2e TO scope_3_tco2e")
print("\u2705 'scope_3_total_tco2e' \u2192 'scope_3_tco2e' umbenannt")
print("Keine Daten bewegt. Parquet-Dateien nutzen interne Feld-IDs, nicht Namen.\n")

spark.sql("DESCRIBE nessie.raw.nzdpu_emissions").show()

---
## Zusammenfassung

Was wir gesehen haben:

**Schema Evolution**: Neue Felder hinzufügen, Felder umbenennen — alles reine Metadaten-Operationen. Bestehende Daten werden nicht verändert, alte Zeilen haben `NULL` für neue Felder.

**Feld-Umbenennung beim Anbieter**: Durch Mapping beim Schreiben bleibt die interne Struktur stabil. Reports brechen nicht.

**Time Travel**: Jede Append-Operation erzeugt einen Snapshot. Der Zustand vor der Änderung ist jederzeit lesbar — per Snapshot-ID, via Spark oder Trino.

**Report-Stabilität**: Ein bestehender Trino-Report läuft nach Schema Evolution unverändert weiter. Neue Felder stören nicht.

**Append-Only**: Iceberg überschreibt keine Dateien. Neue Daten kommen in neuen Parquet-Files. Das ist die Basis für Revisionssicherheit.

---
→ **Notebook 03:** Wie dbt deklarative SQL-Transformationen auf Iceberg macht (Raw → Staging → Curated)